# Homework 3

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import math

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
%%script true # Skips the cell to prevent repeated dataset downloads.

data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!wget $data

In [3]:
df_raw = pd.read_csv('course_lead_scoring_2026.csv')

In [4]:
df = df_raw.copy()
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


## Data Preparation

In [5]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
categorical_features = list(df.dtypes[df.dtypes == 'str'].index)
categorical_features

['lead_source', 'industry', 'employment_status', 'location']

In [8]:
numerical_features = list(df.dtypes[(df.dtypes != 'str') & (df.dtypes.index != 'converted')].index)
numerical_features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [9]:
df[categorical_features] = df[categorical_features].fillna('NA')

df[numerical_features] = df[numerical_features].fillna(0.0)

In [10]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1

##### What is the most frequent observation (mode) for the column industry?

    NA
    technology
    healthcare
    retail

In [11]:
industry_mode = df.industry.mode().iloc[0]

print(f'The most frequent observation mode for the column industry is {industry_mode}.')

The most frequent observation mode for the column industry is technology.


## Question 2

##### Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

###### What are the two features that have the biggest correlation?

    interaction_count and lead_score
    number_of_courses_viewed and lead_score
    number_of_courses_viewed and interaction_count
    annual_income and interaction_count
    
##### Only consider the pairs above when answering this question.

In [12]:
correlation_matrix = df[numerical_features].corr()

correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [13]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    score = correlation_matrix.loc[col1, col2]
    print(f'{col1} and {col2}: {score:.4f}')

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [14]:
strongest_pair = max(
    pairs,
    key=lambda pair: abs(correlation_matrix.loc[pair[0], pair[1]])
)

print('Strongest pair:', strongest_pair)

Strongest pair: ('interaction_count', 'lead_score')


### Data Split

In [15]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

In [16]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [17]:
del df_train['converted']
del df_val['converted']

## Question 3

    Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
    Round the scores to 2 decimals using round(score, 2).
##### Which of these variables has the biggest mutual information score?

    industry
    location
    lead_source
    employment_status

In [18]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

In [19]:
df_mi = df_train[categorical_features].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI')

In [20]:
display(df_mi)

,MI
lead_source,0.034346
employment_status,0.018966
industry,0.001995
location,0.001043


## Question 4

    Now let's train a logistic regression.
    Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
    Fit the model on the training dataset.
        To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
        model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
##### What accuracy did you get?

    0.55
    0.65
    0.75
    0.85

### One-Hot Encoding

In [21]:
def one_hot_encoding(df, dv=None):

    data_dict = df.to_dict(orient='records')

    if dv is None:
        dv = DictVectorizer(sparse=False)
        X = dv.fit_transform(data_dict)
    else:
        X = dv.transform(data_dict)

    return X, dv

In [22]:
def model_train(X, y):
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X, y)
    return model

In [36]:
def model_predict(model, X, y, d=None, verbose=True):
    y_pred = model.predict_proba(X)[:, 1]
    converted = y_pred >= 0.5
    
    accuracy = (y == converted).mean()
    if d is not None:
        accuracy = math.trunc(accuracy * 10**d) / 10**d
        print(f"The accuracy on the validation data rounded to {d} decimal digits is {accuracy:.{d}f}.")
    else:
        if verbose:
            print(f"The accuracy on the validation data without rounding is {accuracy}.")
    return y_pred, accuracy

In [37]:
X_train, dv = one_hot_encoding(df_train)

In [38]:
model = model_train(X_train, y_train)

In [39]:
X_val, _ = one_hot_encoding(df_val, dv)

In [40]:
y_pred, accuracy = model_predict(model, X_val, y_val, 2)

The accuracy on the validation data rounded to 2 decimal digits is 0.64.


## Question 5

    Let's find the least useful feature using the feature elimination technique.
    Train a model using the same features and parameters as in Q4 (without rounding).
    Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
    For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
##### Which of following feature has the smallest difference?

    'lead_source'
    'number_of_courses_viewed'
    'interaction_count'
##### Note: The difference doesn't have to be positive.

In [41]:
df_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NA,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00
...,...,...,...,...,...,...,...,...
2995,paid_ads,finance,employed,asia,81522.0,1,3,0.29
2996,social_media,retail,employed,asia,33075.0,1,1,0.30
2997,organic_search,finance,employed,africa,71532.0,3,5,0.58
2998,social_media,retail,employed,asia,48836.0,0,0,0.03


In [42]:
# Training a model using the same features and parameters as in Q4 (without rounding).

y_pred, accuracy = model_predict(model, X_val, y_val)

The accuracy on the validation data without rounding is 0.645.


In [43]:
df_train_columns = list(df_train.dtypes.index)
df_train_columns

['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [49]:
def feature_elimination(df_train, df_val, y_train, y_val, columns):
    accuracy_dict = {}
    for col in columns:
        df_train_temp = df_train.drop(columns=[col])
        df_val_temp = df_val.drop(columns=[col])

        X_train, dv = one_hot_encoding(df_train_temp)
        model = model_train(X_train, y_train)

        X_val, _ = one_hot_encoding(df_val_temp, dv)
        y_pred, accuracy = model_predict(model, X_val, y_val, verbose=False)
        print()
        print(f"The accuracy of validation dataset without feature '{col}' : {accuracy}")
        print()
        accuracy_dict[f'w/o_{col}'] = accuracy
    
    return accuracy_dict 

In [50]:
accuracy_dict = feature_elimination(df_train, df_val, y_train, y_val, df_train_columns)


The accuracy of validation dataset without feature 'lead_source' : 0.642


The accuracy of validation dataset without feature 'industry' : 0.645


The accuracy of validation dataset without feature 'employment_status' : 0.645


The accuracy of validation dataset without feature 'location' : 0.645


The accuracy of validation dataset without feature 'annual_income' : 0.724


The accuracy of validation dataset without feature 'number_of_courses_viewed' : 0.643


The accuracy of validation dataset without feature 'interaction_count' : 0.601


The accuracy of validation dataset without feature 'lead_score' : 0.644



In [51]:
print(accuracy_dict)

{'w/o_lead_source': np.float64(0.642), 'w/o_industry': np.float64(0.645), 'w/o_employment_status': np.float64(0.645), 'w/o_location': np.float64(0.645), 'w/o_annual_income': np.float64(0.724), 'w/o_number_of_courses_viewed': np.float64(0.643), 'w/o_interaction_count': np.float64(0.601), 'w/o_lead_score': np.float64(0.644)}


In [60]:
df_accuracy = pd.DataFrame(accuracy_dict.items(), columns=['excluded_feature', 'accuracy'])

df_accuracy

,excluded_feature,accuracy
0,w/o_lead_source,0.642
1,w/o_industry,0.645
2,w/o_employment_status,0.645
3,w/o_location,0.645
4,w/o_annual_income,0.724
5,w/o_number_of_courses_viewed,0.643
6,w/o_interaction_count,0.601
7,w/o_lead_score,0.644


In [61]:
df_accuracy['diff'] = accuracy - df_accuracy['accuracy']

df_accuracy

,excluded_feature,accuracy,diff
0,w/o_lead_source,0.642,0.003
1,w/o_industry,0.645,0.000
2,w/o_employment_status,0.645,0.000
3,w/o_location,0.645,0.000
4,w/o_annual_income,0.724,-0.079
5,w/o_number_of_courses_viewed,0.643,0.002
6,w/o_interaction_count,0.601,0.044
7,w/o_lead_score,0.644,0.001


In [79]:
smallest_diff = df_accuracy[
    df_accuracy['excluded_feature'].isin([
        'w/o_lead_source', 'w/o_number_of_courses_viewed', 'w/o_interaction_count'
    ])
]



smallest_diff_answer = smallest_diff.loc[smallest_diff['diff'].idxmin(), 'excluded_feature']

print(f"The feature with smallest differece is {smallest_diff_answer}.")

The feature with smallest differece is w/o_number_of_courses_viewed.


## Question 6

In [90]:
def model_train_reg(X, y, C):
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X, y)
    return model

In [98]:
def model_predict_reg(df_train, df_val, y_train, y_val, C, d=3):
    reg_dict = {}
    for i in C:
        df_train_temp = df_train.copy()
        df_val_temp = df_val.copy()

        X_train, dv = one_hot_encoding(df_train_temp)
        model = model_train_reg(X_train, y_train, i)

        X_val, _ = one_hot_encoding(df_val_temp, dv)
        y_pred, accuracy = model_predict(model, X_val, y_val, verbose=False)
        print()
        print(f"The accuracy of validation dataset when C = {i} : {accuracy}")
        print()
        reg_dict[i] = accuracy
    
    return reg_dict 

In [99]:
c = [0.000001, 0.00001, 0.0001, 0.001]

reg_dict = model_predict_reg(df_train, df_val, y_train, y_val, c)


The accuracy of validation dataset when C = 1e-06 : 0.598


The accuracy of validation dataset when C = 1e-05 : 0.598


The accuracy of validation dataset when C = 0.0001 : 0.613


The accuracy of validation dataset when C = 0.001 : 0.645



In [101]:
df_reg = pd.DataFrame(reg_dict.items(), columns=['reg_strength', 'accuracy'])

df_reg

,reg_strength,accuracy
0,0.000001,0.598
1,0.000010,0.598
2,0.000100,0.613
3,0.001000,0.645


In [103]:
best_reg_acc = df_reg.loc[df_reg['accuracy'].idxmax(), 'reg_strength']

print(f"The C that led to the best accuracy is {best_reg_acc}.")

The C that led to the best accuracy is 0.001.
